<a href="https://colab.research.google.com/github/kadershawa-hub/WF_JAP_Data/blob/main/1D_CNN_example1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Website Fingerprinting Attack on JAP Traffic

This notebook demonstrates a complete Website Fingerprinting attack using the JAP dataset.

**What this notebook covers:**
- Loading the `Filtered_Features.csv` dataset
- Understanding the data structure (signed packet sequences + labels)
- Preprocessing sequences for deep learning (padding, train/test split)
- Building and training a simple CNN model
- Evaluating classification performance

**Author:** [Abdulqader Shaawa]
**Paper:** [A Comprehensive Analysis of Website Fingerprinting on the Java Anon Proxy (JAP) using Machine and Deep Learning]
**Dataset:** [https://github.com/kadershawa-hub/WF\JAP\Data]

---

## 1. Setup and installation

In [ ]:
# Install required packages
!pip install -q tensorflow scikit-learn pandas matplotlib seaborn gdown

# Import libraries
import os
import gdown
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
import time
from tqdm.notebook import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Sequential
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, BatchNormalization, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
print("✅ Packages installed and imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")


## 2. Load and Explore Dataset Structure
The dataset is hosted on Google Drive. We'll download it using gdown.

In [ ]:
print("\n" + "="*60)
print("DOWNLOADING YOUR DATASET")
print("="*60)

file_id = '14WrI5_RcbzCJ8terqemjoLs8Wnsjhf43'
output_file = 'filtered_features.csv'

# Create directory
!mkdir -p data

# Download the filtered dataset using gdown
import gdown
url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, f'data/{output_file}', quiet=False)

# Set data path
DATA_PATH = f'data/{output_file}'

# Verify
if os.path.exists(DATA_PATH):
    file_size = os.path.getsize(DATA_PATH) / (1024 * 1024)  # MB
    print(f"✅ Dataset downloaded: {DATA_PATH}")
    print(f"📊 File size: {file_size:.2f} MB")

    # Quick preview (removed nrows=3 to load full dataset for processing)
    df = pd.read_csv(DATA_PATH)
    print(f"📄 Preview:")
    print(df.head(3))
    print(f"🔤 Columns: {list(df.columns)}")
else:
    print("❌ Download failed! Please check your file ID.")
    # Create synthetic data as fallback
    DATA_PATH = None

print("\n✅ Ready to proceed !")

## STEP 3. CONFIGURE PARAMETERS

In [ ]:
OUTPUT_DIR = 'output'
# Data parameters
SEQUENCE_LENGTH = 3000
NUM_CLASSES = 100
RANDOM_STATE = 42
NUM_FEATURES = 1

DPI = 600
# Training parameters
EPOCHS = 30
BATCH_SIZE = 64
DROPOUT_RATE = 0.3
NORMALIZE_DATA= False
# Output settings

RESULT_DIR = OUTPUT_DIR +"/results"
FIGURES_DIR = OUTPUT_DIR+ "/figures"
# Create output directories if they don't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{RESULT_DIR}", exist_ok=True)
os.makedirs(f"{FIGURES_DIR}", exist_ok=True)
print(f"Figures directory: {FIGURES_DIR}")
print(f"Output directory: {RESULT_DIR}")

print("✅ Configuration loaded!")

## STEP 5. Data Preprocessing for Deep Learning

we need to :
1.   Encode website names to numeric labels
2.   Pad/truncate sequences to a fixed length
3.   Split into train, validation, and test sets




In [ ]:
# Prepare sequence data (signed_packet_lengths)
print("\nParsing signed packet sequences...")
# Extract sequences and labels
sequences = []
labels = []
for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing sequences"):
    # Parse the string representation of the list
    seq_str = row['signed_packet_lengths'].strip('[]')
    seq = [int(x) for x in seq_str.split(',')] if seq_str else []

    # Pad or truncate to fixed length
    if len(seq) > SEQUENCE_LENGTH:
        seq = seq[:SEQUENCE_LENGTH]
    else:
        seq = seq + [0] * (SEQUENCE_LENGTH - len(seq))

    sequences.append(seq)
    labels.append(row['website'])
# convert to numpy arrays
X = np.array(sequences, dtype=np.float32)

# Reshape for 1D CNN/LSTM: (samples, sequence_length, features=1)
X = X.reshape((X.shape[0], X.shape[1], 1))

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
y_cat = to_categorical(y, NUM_CLASSES)

print(f"Data loaded: {X.shape[0]} samples, {len(np.unique(y))} classes")
print(f"Sequence shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")

# Split data into train, validation, and test sets (80%/10%/10%)
print("\nSplitting data...")

X_train, X_temp, y_train_cat, y_temp_cat = train_test_split(
    X, y_cat, test_size=0.2, random_state=RANDOM_STATE, stratify=y_cat)

X_val, X_test, y_val_cat, y_test_cat = train_test_split(
    X_temp, y_temp_cat, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_cat)

print(f"Training set shape: {X_train.shape}, {y_train_cat.shape}")
print(f"Validation set shape: {X_val.shape}, {y_val_cat.shape}")
print(f"Test set shape: {X_test.shape}, {y_test_cat.shape}")

# Optional: Normalize data
if NORMALIZE_DATA:
    print("\nNormalizing data...")
    # Reshape for scaler: (num_samples * sequence_length, 1)
    original_shape_train = X_train.shape
    original_shape_val = X_val.shape
    original_shape_test = X_test.shape

    reshaped_train = X_train.reshape(-1, 1)
    reshaped_val = X_val.reshape(-1, 1)
    reshaped_test = X_test.reshape(-1, 1)

    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_train = scaler.fit_transform(reshaped_train).reshape(original_shape_train)
    X_val = scaler.transform(reshaped_val).reshape(original_shape_val)
    X_test = scaler.transform(reshaped_test).reshape(original_shape_test)
    print("Data normalized.")
else:
    print("Data not normalized.")

## STEP 6. Define the CNN Model


In [ ]:
def build_1d_cnn_model(input_shape, num_classes):
    """
    Build CNN model for website fingerprinting using signed_packet_lengths

    Args:
        input_shape: Tuple (sequence_length, num_features)
        num_classes: Number of websites to classify

    Returns:
        Compiled Keras model
    """
    seq_input = Input(shape=input_shape, name='packet_sequence')

    # First convolutional block
    x = Conv1D(64, 10, activation='relu', padding='same')(seq_input)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.3)(x)

    # Second convolutional block
    x = Conv1D(128, 10, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.3)(x)

    # Third convolutional block
    x = Conv1D(256, 10, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.3)(x)

    # Fully connected layers
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Output layer
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=seq_input, outputs=outputs)

    # Compile model
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-07
    )

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')]
    )

    return model

# Build the model
input_shape = (SEQUENCE_LENGTH, NUM_FEATURES)
model = build_1d_cnn_model(input_shape, NUM_CLASSES)
model.summary()

# ============================================
# 5. Train the Model
# ============================================

print("\n" + "="*50)
print("STEP 3: Training the Model")
print("="*50)

# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model_checkpoint = ModelCheckpoint(
    os.path.join(OUTPUT_DIR, "models", "best_1d_cnn_model.keras"),
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=0.0001
)

start_time = time.time()

history = model.fit(
    X_train, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val_cat),
    callbacks=[early_stopping, model_checkpoint, reduce_lr],
    verbose=1
)

training_time = time.time() - start_time
print(f"\nTraining finished in {training_time:.2f} seconds")

# Load the best model
model = tf.keras.models.load_model(os.path.join(OUTPUT_DIR, "models", "best_1d_cnn_model.keras"))
print(f"Best model loaded from {os.path.join(OUTPUT_DIR, 'models', 'best_1d_cnn_model.keras')}")

# Print best validation accuracy
best_val_accuracy = np.max(history.history['val_accuracy'])
best_epoch = np.argmax(history.history['val_accuracy']) + 1
print(f"Best validation accuracy: {best_val_accuracy:.4f}")
print(f"Epoch with best validation accuracy: {best_epoch}")

In [ ]:
# ============================================
# 6. Evaluate the Model
# ============================================

print("\n" + "="*50)
print("STEP 4: Evaluating the Model")
print("="*50)

# Evaluate on test set
test_loss, test_accuracy, test_top5_accuracy = model.evaluate(
    X_test, y_test_cat, batch_size=BATCH_SIZE
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Top-5 Accuracy: {test_top5_accuracy:.4f}")

# Generate predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

# Classification report
target_names = label_encoder.classes_
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))

# Save classification report
report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_save_path = os.path.join(OUTPUT_DIR, "results", "1d_cnn_classification_report.csv")
report_df.to_csv(report_save_path)
print(f"Classification report saved to {report_save_path}")


In [ ]:
# ============================================
# 7. Visualizations
# ============================================

print("\n" + "="*50)
print("STEP 5: Generating Visualizations")
print("="*50)

# Plot confusion matrix (for top classes only, to keep it readable)
print("\nGenerating confusion matrix...")
top_n = 20
class_counts = df['website'].value_counts()
top_classes = class_counts.head(top_n).index.tolist()
top_indices = [label_encoder.transform([c])[0] for c in top_classes]

# Filter to top classes
mask = [i in top_indices for i in y_true]
filtered_preds = [p for p, m in zip(y_pred, mask) if m]
filtered_labels = [l for l, m in zip(y_true, mask) if m]

cm = confusion_matrix(filtered_labels, filtered_preds)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=top_classes,
    yticklabels=top_classes
)
plt.title(f'Confusion Matrix - 1D CNN (Top {top_n} Classes)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
cm_save_path = os.path.join(FIGURES_DIR, "1d_cnn_confusion_matrix.png")
plt.savefig(cm_save_path, dpi=150)
plt.show()
print(f"Confusion matrix plot saved to {cm_save_path}")

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'], label='Train Accuracy', marker='o', markersize=4)
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='s', markersize=4)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Model Accuracy')
ax1.legend()
ax1.grid(True)

# Loss plot
ax2.plot(history.history['loss'], label='Train Loss', marker='o', markersize=4)
ax2.plot(history.history['val_loss'], label='Validation Loss', marker='s', markersize=4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Model Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
history_plot_path = os.path.join(FIGURES_DIR, "1d_cnn_training_history.png")
plt.savefig(history_plot_path, dpi=150)
plt.show()
print(f"Training history plot saved to {history_plot_path}")